In [5]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

In [6]:
COLS = ["test/nn_class/f1", "test/psd/mean", "test/spd/mean"]
NEW_COLS = {
    "test/nn_class/f1": "1NN-F1",
    "test/psd/mean": "PSD",
    "test/spd/mean": "SPD",
}
MULTIPLIERS = [1, 10**2, 10**2]


def process_csv_file(csv_file: Path):
    results = pd.read_csv(csv_file)
    out = {}
    for col, mul in zip(COLS, MULTIPLIERS, strict=True):
        values = results[col] * mul
        mean_val = values.mean()
        std_val = values.std()
        metric_name = NEW_COLS[col]
        out[metric_name] = f"{mean_val:.3f} ± {std_val:.4f}"
        out[f"{metric_name}_raw"] = mean_val
    return out


def bold_best_results(df: pd.DataFrame, metrics=("1NN-F1", "PSD", "SPD"), maximize=("1NN-F1",)):
    for metric in metrics:
        for dataset, group_idx in df.groupby("Dataset").groups.items():
            subset = df.loc[group_idx]
            raw_col = f"{metric}_raw"
            if metric in maximize:
                best_idx = subset[raw_col].idxmax()
            else:
                best_idx = subset[raw_col].idxmin()

            # Bold the corresponding markdown string
            df.at[best_idx, metric] = f"**{df.at[best_idx, metric]}**"
    return df

# Main Results

In [14]:
def load_main_eval(eval_path: Path) -> pd.DataFrame:
    df_list = []
    for csv_file in sorted(eval_path.iterdir()):
        model, objective, dataset = csv_file.stem.split("_")
        temp_results = process_csv_file(csv_file)
        df_list.append({"Dataset": dataset, "Model": model, "Objective": objective, **temp_results})
    return pd.DataFrame(df_list)

main_results_path = Path("../outputs/eval_main")
main_eval_df = load_main_eval(main_results_path)
main_eval_df = bold_best_results(main_eval_df)

In [15]:
for ds in ["MED", "ABD", "MBA"]:
    print(ds)
    markdown_str = (
        main_eval_df[main_eval_df["Dataset"] == ds][["Model", "Objective", "1NN-F1", "PSD", "SPD"]]
        .sort_values(by=["Model", "Objective"])
        .to_markdown(index=False)
    )
    display(Markdown(markdown_str))

MED


| Model     | Objective   | 1NN-F1             | PSD                | SPD                |
|:----------|:------------|:-------------------|:-------------------|:-------------------|
| NicheFlow | CFM         | 0.609 ± 0.0022     | 0.987 ± 0.0287     | 0.406 ± 0.0039     |
| NicheFlow | GLVFM       | **0.663 ± 0.0021** | **0.895 ± 0.0091** | **0.396 ± 0.0043** |
| NicheFlow | GVFM        | 0.596 ± 0.0023     | 0.990 ± 0.0304     | 0.404 ± 0.0065     |
| RPCFlow   | CFM         | 0.546 ± 0.0012     | 0.976 ± 0.0039     | 0.564 ± 0.0019     |
| RPCFlow   | GLVFM       | 0.589 ± 0.0015     | 0.977 ± 0.0013     | 0.584 ± 0.0021     |
| RPCFlow   | GVFM        | 0.505 ± 0.0005     | 1.165 ± 0.0042     | 0.579 ± 0.0004     |
| SPFlow    | CFM         | 0.273 ± 0.0010     | 1.670 ± 0.0067     | 0.601 ± 0.0006     |
| SPFlow    | GLVFM       | 0.250 ± 0.0004     | 2.245 ± 0.0067     | 0.593 ± 0.0012     |
| SPFlow    | GVFM        | 0.259 ± 0.0010     | 2.359 ± 0.0090     | 0.582 ± 0.0014     |

ABD


| Model     | Objective   | 1NN-F1             | PSD                | SPD                |
|:----------|:------------|:-------------------|:-------------------|:-------------------|
| NicheFlow | CFM         | 0.603 ± 0.0009     | 2.091 ± 0.0057     | **0.567 ± 0.0037** |
| NicheFlow | GLVFM       | **0.628 ± 0.0008** | 2.082 ± 0.0031     | 0.575 ± 0.0049     |
| NicheFlow | GVFM        | 0.573 ± 0.0011     | 2.224 ± 0.0047     | 0.593 ± 0.0030     |
| RPCFlow   | CFM         | 0.524 ± 0.0020     | **2.051 ± 0.0039** | 1.015 ± 0.0036     |
| RPCFlow   | GLVFM       | 0.554 ± 0.0006     | 2.053 ± 0.0044     | 1.038 ± 0.0025     |
| RPCFlow   | GVFM        | 0.477 ± 0.0008     | 2.260 ± 0.0077     | 1.036 ± 0.0031     |
| SPFlow    | CFM         | 0.190 ± 0.0005     | 2.494 ± 0.0051     | 1.119 ± 0.0037     |
| SPFlow    | GLVFM       | 0.173 ± 0.0013     | 2.870 ± 0.0238     | 1.093 ± 0.0037     |
| SPFlow    | GVFM        | 0.175 ± 0.0010     | 3.373 ± 0.0103     | 1.104 ± 0.0023     |

MBA


| Model     | Objective   | 1NN-F1             | PSD                | SPD                |
|:----------|:------------|:-------------------|:-------------------|:-------------------|
| NicheFlow | CFM         | 0.282 ± 0.0005     | 1.557 ± 0.0021     | 0.554 ± 0.0022     |
| NicheFlow | GLVFM       | **0.285 ± 0.0003** | 1.556 ± 0.0018     | 0.532 ± 0.0018     |
| NicheFlow | GVFM        | 0.268 ± 0.0005     | 1.662 ± 0.0032     | **0.532 ± 0.0008** |
| RPCFlow   | CFM         | 0.271 ± 0.0002     | **1.543 ± 0.0017** | 0.811 ± 0.0009     |
| RPCFlow   | GLVFM       | 0.265 ± 0.0005     | 1.723 ± 0.0014     | 0.780 ± 0.0008     |
| RPCFlow   | GVFM        | 0.249 ± 0.0003     | 1.750 ± 0.0020     | 0.784 ± 0.0008     |
| SPFlow    | CFM         | 0.205 ± 0.0003     | 1.834 ± 0.0019     | 0.824 ± 0.0012     |
| SPFlow    | GLVFM       | 0.194 ± 0.0003     | 2.322 ± 0.0034     | 0.853 ± 0.0012     |
| SPFlow    | GVFM        | 0.181 ± 0.0003     | 2.583 ± 0.0032     | 0.834 ± 0.0012     |

# Ablations

### $K$ Regions Ablation Study 

In [16]:
def load_kregion_ablation(eval_path: Path, base_path: Path) -> pd.DataFrame:
    df_list = []
    for csv_file in sorted(eval_path.iterdir()):
        model, objective, dataset, kregions = csv_file.stem.split("_")
        kregions = int(kregions.replace("KRegions=", ""))
        temp_results = process_csv_file(csv_file)
        df_list.append({"Dataset": dataset, "K": kregions, **temp_results})

    for csv_file in sorted(base_path.iterdir()):
        if "NicheFlow_GLVFM" not in csv_file.stem:
            continue
        _, _, dataset = csv_file.stem.split("_")
        temp_results = process_csv_file(csv_file)
        df_list.append({"Dataset": dataset, "K": 64, **temp_results})

    return pd.DataFrame(df_list)


kregions_results_path = Path("../outputs/eval_kregion_ablations")
kregion_df = load_kregion_ablation(kregions_results_path, main_results_path)
kregion_df = bold_best_results(kregion_df)

In [17]:
display(
    Markdown(
        kregion_df[["Dataset", "K", *list(NEW_COLS.values())]]
        .sort_values(by=["Dataset", "K"])
        .to_markdown(index=False)
    )
)

| Dataset   |   K | 1NN-F1             | PSD                | SPD                |
|:----------|----:|:-------------------|:-------------------|:-------------------|
| ABD       |   8 | 0.617 ± 0.0007     | 1.955 ± 0.0028     | 0.629 ± 0.0056     |
| ABD       |  16 | **0.633 ± 0.0015** | **1.938 ± 0.0051** | 0.624 ± 0.0028     |
| ABD       |  32 | 0.623 ± 0.0009     | 1.969 ± 0.0033     | 0.640 ± 0.0033     |
| ABD       |  64 | 0.628 ± 0.0008     | 2.082 ± 0.0031     | **0.575 ± 0.0049** |
| MBA       |   8 | 0.283 ± 0.0001     | 1.562 ± 0.0012     | 0.572 ± 0.0025     |
| MBA       |  16 | 0.283 ± 0.0003     | 1.565 ± 0.0012     | 0.540 ± 0.0019     |
| MBA       |  32 | 0.279 ± 0.0003     | 1.601 ± 0.0022     | 0.538 ± 0.0013     |
| MBA       |  64 | **0.285 ± 0.0003** | **1.556 ± 0.0018** | **0.532 ± 0.0018** |
| MED       |   8 | 0.639 ± 0.0029     | **0.874 ± 0.0075** | 0.440 ± 0.0066     |
| MED       |  16 | 0.660 ± 0.0017     | 0.879 ± 0.0070     | 0.394 ± 0.0031     |
| MED       |  32 | 0.658 ± 0.0019     | 0.896 ± 0.0096     | **0.390 ± 0.0072** |
| MED       |  64 | **0.663 ± 0.0021** | 0.895 ± 0.0091     | 0.396 ± 0.0043     |

## $\lambda$ OT Ablations

In [11]:
def load_lambda_ablation(eval_path: Path, base_path: Path) -> pd.DataFrame:
    df_list = []
    for csv_file in sorted(eval_path.iterdir()):
        model, objective, dataset, ot_lambda = csv_file.stem.split("_")
        ot_lambda = float(ot_lambda.replace("OTLambda=", ""))
        temp_results = process_csv_file(csv_file)
        df_list.append(
            {
                "Dataset": dataset,
                "Model": model,
                "Objective": objective,
                "Lambda": ot_lambda,
                **temp_results,
            }
        )

    for csv_file in sorted(base_path.iterdir()):
        if not ("NicheFlow" in csv_file.stem or "RPCFlow" in csv_file.stem):
            continue
        model, objective, dataset = csv_file.stem.split("_")
        temp_results = process_csv_file(csv_file)
        df_list.append(
            {
                "Dataset": dataset,
                "Model": model,
                "Objective": objective,
                "Lambda": 0.1,
                **temp_results,
            }
        )

    return pd.DataFrame(df_list).sort_values(by=["Model", "Objective", "Lambda"])


ot_results_path = Path("../outputs/eval_ot_ablations")
lambda_df = load_lambda_ablation(ot_results_path, main_results_path)
lambda_df = bold_best_results(lambda_df)

In [12]:
for ds in ["MED", "ABD", "MBA"]:
    print(ds)
    markdown_str = (
        lambda_df[lambda_df["Dataset"] == ds][
            ["Model", "Objective", "Lambda", *list(NEW_COLS.values())]
        ]
        .sort_values(by=["Model", "Objective", "Lambda"])
        .to_markdown(index=False)
    )
    display(Markdown(markdown_str))

MED


| Model     | Objective   |   Lambda | 1NN-F1             | PSD                | SPD                |
|:----------|:------------|---------:|:-------------------|:-------------------|:-------------------|
| NicheFlow | CFM         |     0.1  | 0.609 ± 0.0022     | 0.987 ± 0.0287     | 0.406 ± 0.0039     |
| NicheFlow | CFM         |     0.25 | 0.569 ± 0.0030     | 0.972 ± 0.0340     | 0.432 ± 0.0053     |
| NicheFlow | CFM         |     0.5  | 0.549 ± 0.0026     | 1.051 ± 0.0288     | 0.476 ± 0.0076     |
| NicheFlow | CFM         |     0.75 | 0.516 ± 0.0034     | 1.130 ± 0.0580     | 0.522 ± 0.0092     |
| NicheFlow | GLVFM       |     0.1  | **0.663 ± 0.0021** | 0.895 ± 0.0091     | **0.396 ± 0.0043** |
| NicheFlow | GLVFM       |     0.25 | 0.627 ± 0.0019     | 0.918 ± 0.0127     | 0.397 ± 0.0072     |
| NicheFlow | GLVFM       |     0.5  | 0.609 ± 0.0044     | 0.906 ± 0.0087     | 0.426 ± 0.0078     |
| NicheFlow | GLVFM       |     0.75 | 0.590 ± 0.0005     | **0.883 ± 0.0047** | 0.475 ± 0.0194     |
| NicheFlow | GVFM        |     0.1  | 0.596 ± 0.0023     | 0.990 ± 0.0304     | 0.404 ± 0.0065     |
| NicheFlow | GVFM        |     0.25 | 0.564 ± 0.0022     | 1.066 ± 0.0268     | 0.493 ± 0.0099     |
| NicheFlow | GVFM        |     0.5  | 0.535 ± 0.0050     | 1.026 ± 0.0180     | 0.798 ± 0.0571     |
| NicheFlow | GVFM        |     0.75 | 0.524 ± 0.0045     | 1.108 ± 0.0228     | 0.882 ± 0.0636     |
| RPCFlow   | CFM         |     0.1  | 0.546 ± 0.0012     | 0.976 ± 0.0039     | 0.564 ± 0.0019     |
| RPCFlow   | CFM         |     0.25 | 0.546 ± 0.0006     | 0.952 ± 0.0038     | 0.569 ± 0.0014     |
| RPCFlow   | CFM         |     0.5  | 0.554 ± 0.0010     | 0.982 ± 0.0038     | 0.564 ± 0.0007     |
| RPCFlow   | CFM         |     0.75 | 0.539 ± 0.0012     | 0.976 ± 0.0036     | 0.594 ± 0.0012     |
| RPCFlow   | GLVFM       |     0.1  | 0.589 ± 0.0015     | 0.977 ± 0.0013     | 0.584 ± 0.0021     |
| RPCFlow   | GLVFM       |     0.25 | 0.595 ± 0.0007     | 0.925 ± 0.0026     | 0.575 ± 0.0012     |
| RPCFlow   | GLVFM       |     0.5  | 0.589 ± 0.0016     | 0.935 ± 0.0018     | 0.568 ± 0.0010     |
| RPCFlow   | GLVFM       |     0.75 | 0.596 ± 0.0014     | 0.947 ± 0.0030     | 0.570 ± 0.0014     |
| RPCFlow   | GVFM        |     0.1  | 0.505 ± 0.0005     | 1.165 ± 0.0042     | 0.579 ± 0.0004     |
| RPCFlow   | GVFM        |     0.25 | 0.523 ± 0.0007     | 1.223 ± 0.0050     | 0.566 ± 0.0012     |
| RPCFlow   | GVFM        |     0.5  | 0.522 ± 0.0011     | 1.177 ± 0.0047     | 0.569 ± 0.0011     |
| RPCFlow   | GVFM        |     0.75 | 0.516 ± 0.0012     | 1.194 ± 0.0019     | 0.574 ± 0.0016     |

ABD


| Model     | Objective   |   Lambda | 1NN-F1             | PSD                | SPD                |
|:----------|:------------|---------:|:-------------------|:-------------------|:-------------------|
| NicheFlow | CFM         |     0.1  | 0.603 ± 0.0009     | 2.091 ± 0.0057     | 0.567 ± 0.0037     |
| NicheFlow | CFM         |     0.25 | 0.586 ± 0.0023     | 2.112 ± 0.0055     | **0.567 ± 0.0021** |
| NicheFlow | CFM         |     0.5  | 0.585 ± 0.0019     | 2.091 ± 0.0052     | 0.595 ± 0.0041     |
| NicheFlow | CFM         |     0.75 | 0.572 ± 0.0012     | 2.136 ± 0.0086     | 0.596 ± 0.0032     |
| NicheFlow | GLVFM       |     0.1  | **0.628 ± 0.0008** | 2.082 ± 0.0031     | 0.575 ± 0.0049     |
| NicheFlow | GLVFM       |     0.25 | 0.619 ± 0.0014     | 2.103 ± 0.0029     | 0.582 ± 0.0044     |
| NicheFlow | GLVFM       |     0.5  | 0.610 ± 0.0016     | 2.139 ± 0.0041     | 0.579 ± 0.0071     |
| NicheFlow | GLVFM       |     0.75 | 0.603 ± 0.0010     | 2.107 ± 0.0062     | 0.596 ± 0.0038     |
| NicheFlow | GVFM        |     0.1  | 0.573 ± 0.0011     | 2.224 ± 0.0047     | 0.593 ± 0.0030     |
| NicheFlow | GVFM        |     0.25 | 0.571 ± 0.0012     | 2.353 ± 0.0097     | 0.622 ± 0.0091     |
| NicheFlow | GVFM        |     0.5  | 0.557 ± 0.0015     | 2.290 ± 0.0072     | 0.750 ± 0.0081     |
| NicheFlow | GVFM        |     0.75 | 0.556 ± 0.0028     | 2.166 ± 0.0076     | 0.688 ± 0.0121     |
| RPCFlow   | CFM         |     0.1  | 0.524 ± 0.0020     | 2.051 ± 0.0039     | 1.015 ± 0.0036     |
| RPCFlow   | CFM         |     0.25 | 0.511 ± 0.0015     | 2.079 ± 0.0053     | 1.024 ± 0.0046     |
| RPCFlow   | CFM         |     0.5  | 0.507 ± 0.0011     | 2.077 ± 0.0065     | 1.023 ± 0.0032     |
| RPCFlow   | CFM         |     0.75 | 0.517 ± 0.0005     | 2.033 ± 0.0072     | 1.026 ± 0.0050     |
| RPCFlow   | GLVFM       |     0.1  | 0.554 ± 0.0006     | 2.053 ± 0.0044     | 1.038 ± 0.0025     |
| RPCFlow   | GLVFM       |     0.25 | 0.555 ± 0.0013     | 2.076 ± 0.0057     | 1.036 ± 0.0024     |
| RPCFlow   | GLVFM       |     0.5  | 0.551 ± 0.0007     | 2.038 ± 0.0045     | 1.032 ± 0.0034     |
| RPCFlow   | GLVFM       |     0.75 | 0.561 ± 0.0008     | **2.014 ± 0.0056** | 1.035 ± 0.0047     |
| RPCFlow   | GVFM        |     0.1  | 0.477 ± 0.0008     | 2.260 ± 0.0077     | 1.036 ± 0.0031     |
| RPCFlow   | GVFM        |     0.25 | 0.478 ± 0.0014     | 2.364 ± 0.0065     | 1.030 ± 0.0042     |
| RPCFlow   | GVFM        |     0.5  | 0.480 ± 0.0008     | 2.360 ± 0.0036     | 1.025 ± 0.0015     |
| RPCFlow   | GVFM        |     0.75 | 0.471 ± 0.0012     | 2.476 ± 0.0082     | 1.037 ± 0.0036     |

MBA


| Model     | Objective   |   Lambda | 1NN-F1             | PSD                | SPD                |
|:----------|:------------|---------:|:-------------------|:-------------------|:-------------------|
| NicheFlow | CFM         |     0.1  | 0.282 ± 0.0005     | 1.557 ± 0.0021     | 0.554 ± 0.0022     |
| NicheFlow | CFM         |     0.25 | 0.281 ± 0.0006     | 1.546 ± 0.0014     | 0.613 ± 0.0016     |
| NicheFlow | CFM         |     0.5  | 0.283 ± 0.0007     | 1.589 ± 0.0032     | 0.603 ± 0.0024     |
| NicheFlow | CFM         |     0.75 | 0.278 ± 0.0004     | 1.566 ± 0.0043     | 0.615 ± 0.0016     |
| NicheFlow | GLVFM       |     0.1  | **0.285 ± 0.0003** | 1.556 ± 0.0018     | 0.532 ± 0.0018     |
| NicheFlow | GLVFM       |     0.25 | 0.283 ± 0.0002     | 1.598 ± 0.0022     | 0.552 ± 0.0028     |
| NicheFlow | GLVFM       |     0.5  | 0.282 ± 0.0006     | 1.599 ± 0.0031     | 0.546 ± 0.0018     |
| NicheFlow | GLVFM       |     0.75 | 0.281 ± 0.0006     | 1.572 ± 0.0026     | 0.593 ± 0.0032     |
| NicheFlow | GVFM        |     0.1  | 0.268 ± 0.0005     | 1.662 ± 0.0032     | **0.532 ± 0.0008** |
| NicheFlow | GVFM        |     0.25 | 0.266 ± 0.0004     | 1.598 ± 0.0036     | 0.591 ± 0.0030     |
| NicheFlow | GVFM        |     0.5  | 0.269 ± 0.0006     | 1.607 ± 0.0033     | 0.609 ± 0.0028     |
| NicheFlow | GVFM        |     0.75 | 0.263 ± 0.0005     | 1.611 ± 0.0032     | 0.730 ± 0.0061     |
| RPCFlow   | CFM         |     0.1  | 0.271 ± 0.0002     | 1.543 ± 0.0017     | 0.811 ± 0.0009     |
| RPCFlow   | CFM         |     0.25 | 0.272 ± 0.0004     | **1.535 ± 0.0008** | 0.818 ± 0.0005     |
| RPCFlow   | CFM         |     0.5  | 0.273 ± 0.0002     | 1.546 ± 0.0009     | 0.811 ± 0.0019     |
| RPCFlow   | CFM         |     0.75 | 0.275 ± 0.0004     | 1.553 ± 0.0005     | 0.802 ± 0.0013     |
| RPCFlow   | GLVFM       |     0.1  | 0.265 ± 0.0005     | 1.723 ± 0.0014     | 0.780 ± 0.0008     |
| RPCFlow   | GLVFM       |     0.25 | 0.267 ± 0.0002     | 1.728 ± 0.0013     | 0.777 ± 0.0008     |
| RPCFlow   | GLVFM       |     0.5  | 0.269 ± 0.0004     | 1.716 ± 0.0014     | 0.783 ± 0.0006     |
| RPCFlow   | GLVFM       |     0.75 | 0.268 ± 0.0004     | 1.674 ± 0.0019     | 0.785 ± 0.0007     |
| RPCFlow   | GVFM        |     0.1  | 0.249 ± 0.0003     | 1.750 ± 0.0020     | 0.784 ± 0.0008     |
| RPCFlow   | GVFM        |     0.25 | 0.246 ± 0.0005     | 1.708 ± 0.0014     | 0.788 ± 0.0008     |
| RPCFlow   | GVFM        |     0.5  | 0.245 ± 0.0004     | 1.757 ± 0.0026     | 0.774 ± 0.0013     |
| RPCFlow   | GVFM        |     0.75 | 0.247 ± 0.0004     | 1.825 ± 0.0017     | 0.781 ± 0.0009     |